<a href="https://colab.research.google.com/github/serahnjogu-new/AI4EAC-Climate-Health-Risk-Prediction-Challenge-/blob/main/0_82297.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

In [8]:
# 1. LOAD DATA
train = pd.read_csv('/content/Train.csv')
test = pd.read_csv('/content/Test.csv')
climate = pd.read_csv('/content/climate_features.csv')
ss = pd.read_csv('/content/SampleSubmission.csv')

In [9]:
# 2. MERGE CLIMATE DATA
# Joining supplemental climate features to the main datasets
train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

In [10]:
# 3. FEATURE ENGINEERING
def engineer_features(df):
    # Convert dates
    df['deathdate'] = pd.to_datetime(df['deathdate'])

    # Temporal Features (Seasonality is key for climate health)
    df['month'] = df['deathdate'].dt.month
    df['day_of_year'] = df['deathdate'].dt.dayofyear
    df['year'] = df['deathdate'].dt.year

    # Climate Interaction: Temperature Range
    # Extreme fluctuations often impact health more than steady heat
    df['temp_diff_30d'] = df['tmax_30d'] - df['tmin_30d']

    # Rainfall Intensity
    # Identifying if rain was concentrated in a short burst vs spread out
    df['rain_intensity'] = df['rain_sum_30d'] / (df['rain_days_30d'] + 1)

    # Categorical Encoding
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone'] = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    # Drop columns that won't be used in the model
    return df.drop(['ID', 'deathdate', 'location', 'deathdate_extra'], axis=1, errors='ignore')

train_df = engineer_features(train)
test_df = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1)
y = train_df['is_climate_sensitive']
X_test = test_df.copy()

In [11]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# ... (Previous code for loading and merging) ...

# 3.5 HANDLE MISSING VALUES
# Fill missing values with median - crucial for the extra climate columns
X = X.fillna(X.median())
X_test = X_test.fillna(X_test.median())

# 4. CORRECTED CROSS-VALIDATION
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oob_probs = np.zeros(len(X))
test_probs = np.zeros(len(X_test))

print("Starting Training...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Define the model
    model = LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        scale_pos_weight=2.5, # Increased weight for climate-sensitive class
        random_state=42,
        importance_type='gain'
    )

    # Fit using the new callback system
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            early_stopping(stopping_rounds=100),
            log_evaluation(period=0) # Set period > 0 to see progress
        ]
    )

    oob_probs[val_idx] = model.predict_proba(X_val)[:, 1]
    test_probs += model.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"Fold {fold+1} complete.")

# ... (Rest of evaluation and submission code) ...

Starting Training...
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001151 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5114
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

In [12]:
# 5. EVALUATION
threshold = 0.5
oob_preds = (oob_probs >= threshold).astype(int)

f1 = f1_score(y, oob_preds)
auc = roc_auc_score(y, oob_probs)
final_score = (0.5 * f1) + (0.5 * auc)

print(f"OOB F1 Score: {f1:.4f}")
print(f"OOB AUC Score: {auc:.4f}")
print(f"Combined Competition Score: {final_score:.4f}")

OOB F1 Score: 0.7998
OOB AUC Score: 0.8058
Combined Competition Score: 0.8028


In [13]:
#6. FINAL SUBMISSION
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_probs >= 0.5).astype(int),
    'TargetRAUC': test_probs
})

submission.to_csv('winning_submission.csv', index=False)
print("Submission file saved!")

Submission file saved!


In [16]:
from google.colab import files
files.download('winning_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# 1. LOAD DATA
train = pd.read_csv('/content/Train.csv')
test  = pd.read_csv('/content/Test.csv')
climate = pd.read_csv('/content/climate_features.csv')
ss = pd.read_csv('/content/SampleSubmission.csv')

# 2. MERGE CLIMATE DATA
train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 3. FEATURE ENGINEERING
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])

    # Temporal
    df['month']       = df['deathdate'].dt.month
    df['day_of_year'] = df['deathdate'].dt.dayofyear
    df['year']        = df['deathdate'].dt.year
    df['season']      = df['month'].map({
        12:0, 1:0, 2:0,
        3:1,  4:1, 5:1,
        6:2,  7:2, 8:2,
        9:3, 10:3, 11:3
    })

    # Climate interactions — the ones that worked!
    df['temp_diff_30d']  = df['tmax_30d'] - df['tmin_30d']
    df['rain_intensity'] = df['rain_sum_30d'] / (df['rain_days_30d'] + 1)
    df['temp_range']     = df['max_temperature'] - df['min_temperature']

    # NEW: more climate features
    df['rain_intensity_7d']  = df['rain_sum_7d'] / 7
    df['rain_intensity_90d'] = df['rain_sum_90d'] / (90 + 1)
    df['temp_stress']        = df['tmax_30d'] * df['rain_sum_30d']
    df['ndvi_change']        = df['ndvi_30d'] - df['ndvi_90d']

    # Age — most important feature
    df['age_group']  = pd.cut(df['age'], bins=[-1,1,5,15,60,200], labels=[0,1,2,3,4]).astype(int)
    df['is_infant']  = (df['age'] <= 1).astype(int)
    df['is_child']   = (df['age'] <= 5).astype(int)
    df['is_elderly'] = (df['age'] >= 60).astype(int)
    df['log_age']    = np.log1p(df['age'])

    # Interactions
    df['age_x_precip']   = df['age'] * df['precipitation']
    df['age_x_rain30d']  = df['age'] * df['rain_sum_30d']
    df['young_rainy']    = ((df['age'] <= 5) & (df['precipitation'] > 0)).astype(int)
    df['infant_hot']     = ((df['age'] <= 1) & (df['tmax_30d'] > 28)).astype(int)
    df['age_x_season']   = df['age'] * df['season']

    # Categorical encoding
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    return df.drop(['ID', 'deathdate', 'location', 'deathdate_extra'], axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X      = train_df.drop('is_climate_sensitive', axis=1)
y      = train_df['is_climate_sensitive']
X_test = test_df.copy()

# Fill missing values
X      = X.fillna(X.median())
X_test = X_test.fillna(X_test.median())

print("X shape:", X.shape)

# 4. OOF TRAINING — LGBM (proven best) + XGB
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgbm  = np.zeros(len(X))
oof_xgb   = np.zeros(len(X))
test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))

print("Training LGBM...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm = LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        scale_pos_weight=2.5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        importance_type='gain'
    )
    lgbm.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[early_stopping(100), log_evaluation(0)]
    )
    oof_lgbm[val_idx]  = lgbm.predict_proba(X_val)[:, 1]
    test_lgbm         += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"  Fold {fold+1} done")

print("\nTraining XGB...")
neg = (y==0).sum(); pos = (y==1).sum()
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    xgb = XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        scale_pos_weight=neg/pos,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=100,
        eval_metric='auc',
        random_state=42,
        verbosity=0
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    test_xgb         += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"  Fold {fold+1} done")

# 5. ENSEMBLE OOF
oof_ensemble  = (oof_lgbm + oof_xgb) / 2
test_ensemble = (test_lgbm + test_xgb) / 2

# 6. EVALUATE
f1_lgbm  = f1_score(y, (oof_lgbm >= 0.5).astype(int))
auc_lgbm = roc_auc_score(y, oof_lgbm)
f1_xgb   = f1_score(y, (oof_xgb >= 0.5).astype(int))
auc_xgb  = roc_auc_score(y, oof_xgb)
f1_ens   = f1_score(y, (oof_ensemble >= 0.5).astype(int))
auc_ens  = roc_auc_score(y, oof_ensemble)

print("\n=== RESULTS ===")
print(f"LGBM:     F1={f1_lgbm:.4f}  AUC={auc_lgbm:.4f}  Score={0.6*f1_lgbm+0.4*auc_lgbm:.4f}")
print(f"XGB:      F1={f1_xgb:.4f}  AUC={auc_xgb:.4f}  Score={0.6*f1_xgb+0.4*auc_xgb:.4f}")
print(f"Ensemble: F1={f1_ens:.4f}  AUC={auc_ens:.4f}  Score={0.6*f1_ens+0.4*auc_ens:.4f}")

# 7. SUBMISSION — use best OOF model
best_probs = test_ensemble  # change to test_lgbm if ensemble is worse

submission = pd.DataFrame({
    'ID':         ss['ID'],
    'TargetF1':   (best_probs >= 0.5).astype(int),
    'TargetRAUC': best_probs
})

submission.to_csv('submission_improved.csv', index=False)
print("\nTargetF1 distribution:")
print(submission['TargetF1'].value_counts())

from google.colab import files
files.download('submission_improved.csv')

X shape: (3146, 46)
Training LGBM...
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001614 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7168
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Use LGBM only — proven winner
# And fix the over-prediction by using LGBM test probs only

submission = pd.DataFrame({
    'ID':         ss['ID'],
    'TargetF1':   (test_lgbm >= 0.5).astype(int),
    'TargetRAUC': test_lgbm
})

submission.to_csv('submission_lgbm_only.csv', index=False)
print("LGBM only distribution:")
print(submission['TargetF1'].value_counts())

from google.colab import files
files.download('submission_lgbm_only.csv')

LGBM only distribution:
TargetF1
1    1001
0      29
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# 1. LOAD DATA
train = pd.read_csv('/content/Train.csv')
test  = pd.read_csv('/content/Test.csv')
climate = pd.read_csv('/content/climate_features.csv')
ss = pd.read_csv('/content/SampleSubmission.csv')

# 2. MERGE
train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 3. FEATURE ENGINEERING
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])

    # Temporal
    df['month']       = df['deathdate'].dt.month
    df['day_of_year'] = df['deathdate'].dt.dayofyear
    df['year']        = df['deathdate'].dt.year
    df['season']      = df['month'].map({
        12:0, 1:0, 2:0,
        3:1,  4:1, 5:1,
        6:2,  7:2, 8:2,
        9:3, 10:3, 11:3
    })

    # Climate features
    df['temp_diff_30d']    = df['tmax_30d'] - df['tmin_30d']
    df['rain_intensity']   = df['rain_sum_30d'] / (df['rain_days_30d'] + 1)
    df['temp_range']       = df['max_temperature'] - df['min_temperature']
    df['rain_intensity_7d']  = df['rain_sum_7d'] / 7
    df['rain_intensity_90d'] = df['rain_sum_90d'] / 91
    df['temp_stress']        = df['tmax_30d'] * df['rain_sum_30d']
    df['ndvi_change']        = df['ndvi_30d'] - df['ndvi_90d']
    df['hot_rain_days']      = df['hot_days_30d'] * df['rain_days_30d']
    df['elevation_x_rain']   = df['elevation'] * df['rain_sum_30d']
    df['tmax_x_ndvi']        = df['tmax_30d'] * df['ndvi_30d']

    # Age
    df['age_group']  = pd.cut(df['age'], bins=[-1,1,5,15,60,200], labels=[0,1,2,3,4]).astype(int)
    df['is_infant']  = (df['age'] <= 1).astype(int)
    df['is_child']   = (df['age'] <= 5).astype(int)
    df['is_elderly'] = (df['age'] >= 60).astype(int)
    df['log_age']    = np.log1p(df['age'])

    # Interactions
    df['age_x_precip']      = df['age'] * df['precipitation']
    df['age_x_rain30d']     = df['age'] * df['rain_sum_30d']
    df['age_x_tmax']        = df['age'] * df['tmax_30d']
    df['young_rainy']       = ((df['age'] <= 5) & (df['precipitation'] > 0)).astype(int)
    df['infant_hot']        = ((df['age'] <= 1) & (df['tmax_30d'] > 28)).astype(int)
    df['age_x_season']      = df['age'] * df['season']
    df['child_high_rain']   = ((df['age'] <= 5) & (df['rain_sum_30d'] > 50)).astype(int)
    df['elderly_hot']       = ((df['age'] >= 60) & (df['tmax_30d'] > 30)).astype(int)

    # Categorical
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    return df.drop(['ID', 'deathdate', 'location', 'deathdate_extra'], axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X      = train_df.drop('is_climate_sensitive', axis=1)
y      = train_df['is_climate_sensitive']
X_test = test_df.copy()

X      = X.fillna(X.median())
X_test = X_test.fillna(X_test.median())

print("X shape:", X.shape)

# 4. OOF TRAINING
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg = (y==0).sum(); pos = (y==1).sum()

oof_lgbm  = np.zeros(len(X))
oof_xgb   = np.zeros(len(X))
oof_rf    = np.zeros(len(X))
test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_rf   = np.zeros(len(X_test))

# LGBM
print("Training LGBM...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=7,
        num_leaves=63,
        scale_pos_weight=2.5,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        importance_type='gain'
    )
    model.fit(X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[early_stopping(150), log_evaluation(0)]
    )
    oof_lgbm[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_lgbm         += model.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"  Fold {fold+1} done")

# XGB
print("\nTraining XGB...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = XGBClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=7,
        scale_pos_weight=neg/pos,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        early_stopping_rounds=150,
        eval_metric='auc',
        random_state=42,
        verbosity=0
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_xgb         += model.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"  Fold {fold+1} done")

# RF
print("\nTraining RF...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    oof_rf[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_rf         += model.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"  Fold {fold+1} done")

# 5. ENSEMBLE
oof_ensemble  = (oof_lgbm + oof_xgb + oof_rf) / 3
test_ensemble = (test_lgbm + test_xgb + test_rf) / 3

# 6. EVALUATE ALL
print("\n=== RESULTS ===")
for name, oof, test_p in [
    ('LGBM', oof_lgbm, test_lgbm),
    ('XGB',  oof_xgb,  test_xgb),
    ('RF',   oof_rf,   test_rf),
    ('Ensemble', oof_ensemble, test_ensemble)
]:
    f1  = f1_score(y, (oof >= 0.5).astype(int))
    auc = roc_auc_score(y, oof)
    print(f"{name:10}: F1={f1:.4f}  AUC={auc:.4f}  Score={0.6*f1+0.4*auc:.4f}")

# 7. SUBMISSION
submission = pd.DataFrame({
    'ID':         ss['ID'],
    'TargetF1':   (test_ensemble >= 0.5).astype(int),
    'TargetRAUC': test_ensemble
})

submission.to_csv('submission_v9.csv', index=False)
print("\nTargetF1 distribution:")
print(submission['TargetF1'].value_counts())

from google.colab import files
files.download('submission_v9.csv')

X shape: (3146, 52)
Training LGBM...
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000584 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7937
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 150 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit